Circuit format:
    {"n_qubits": 2, "gates": [{"gate": "H", "qubits": [0]}, ...]}

Supported gates: H, X, Z, S, SDG, T, TDG, CX.
CX qubit order: [control, target]. Qubit 0 is the top/MSB wire.
Gates execute in listed order. One through six qubits are supported.

Config format:
    {"w1": 1.0, "w2": 6.0, "wd": 0.2}

This includes a LOCAL checker, not the organizers' private tests.

In [ ]:
import copy
import math

import numpy as np


SINGLE_QUBIT_GATES = frozenset(("H", "X", "Z", "S", "SDG", "T", "TDG"))
ALLOWED_GATES = SINGLE_QUBIT_GATES | {"CX"}




SELF_INVERSE = {"H", "X", "Z", "CX"}

INVERSE_PAIRS = {
    ("S", "SDG"),
    ("SDG", "S"),
    ("T", "TDG"),
    ("TDG", "T"),
}

def can_cancel_redundancies(last_gate, next_gate):
    """Return True if two gates cancel."""

    # if this gate = last gate and they're their own inverse
    if last_gate == next_gate and last_gate in SELF_INVERSE:
        return True
    # if this gate = inverse of last or vice versa
    if (last_gate, next_gate) in INVERSE_PAIRS:
        return True
    # otherwise
    return False

def replacement_gate(last_gate, next_gate):
    if last_gate == next_gate == "S":
        return "Z"
    if last_gate == next_gate == "T":
        return "S"
    if last_gate == next_gate == "SDG":
        return "Z"
    if last_gate == next_gate == "TDG":
        return "SDG"
    return None


def optimize(circuit, config):
    """Return an equivalent circuit (EDIT THIS FUNCTION).
    
        You must not mutate `circuit` or `config`. The starter returns
        an unchanged copy, which is valid but earns no optimization gain.
    """
    # Checkers
    validate_circuit(circuit)
    validate_config(config)



    n_qubits = circuit["n_qubits"] # stores n_qubits int from circuit
    gates = copy.deepcopy(circuit["gates"]) # stores gates dictionary from circuit

    # stacks for each qubit (lane)
    qubit_stacks = [[] for _ in range(n_qubits)]

    # Track which original gates have been canceled
    removed = set()

    for gate_index, gate_info in enumerate(gates):

        gate_type = gate_info["gate"] # the abbreviation of the gate (H, CX, Y, etc.)
        qubits = gate_info["qubits"] # which qubits were listed to apply the gate for this iteration to ([0,1], [1], [1,2,3], etc.)

        # Check every qubit this gate acts on
        can_cancel = True # a boolean tracker for whether the current gate can cancel with the previous gate in this lane
        can_replace = True # a boolean tracker for whether the current gate is an S/T and matches another S/T to yeild a Z/S
        previous_index = None # begins as none because we start at beginning = nothing prior yet
        # qubits is a list like [0,1] telling us said gate is applied to qubits 0 and 1. So for each qubit stack (ie: [0,1] would be the stacks for 0 and 1
        # that THIS gate is being applied to, which allows us to apply a CNOT gate to both its target and control qubits )
        for qubit_stack in qubits:

            # sets a reference pointing to the pre-existing stack this qubit 
            current_stack = qubit_stacks[qubit_stack]

            # if the stack is empty, theres no gate to possibly cancel, and nothing that can be replaced
            if len(current_stack) == 0:
                can_cancel = False 
                can_replace = False
                break # breaks because otherwise next line, current_stack[-1] would throw an IndexError

            last_gate_index = current_stack[-1] # Recall, stored here in the stack is the index for which THIS GATE belongs in the ORIGINAL gates ARRAY--the most recent gate this qubit saw
            last_gate_info = gates[last_gate_index] # Therefore, we can now retrieve the full line of info given to us about this gate

            # Gates must act on exactly the same qubits
            if last_gate_info["qubits"] != qubits: # If the last gate's qubit list doesn't match the current gate's qubit list, obviously they're not being applied to the same qubit = different lanes
                can_cancel = False # Thus it can't cancel it out
                can_replace = False
                break

            # Check whether gate types cancel
            cancel = can_cancel_redundancies(last_gate_info["gate"], gate_type)
            replacement = replacement_gate(last_gate_info["gate"], gate_type)
            if not cancel and replacement is None:
                can_cancel = False
                can_replace = False
                break

            if not cancel:
                can_cancel = False

            if replacement is None:
                can_replace = False

            # All stacks must point to the same previous operation
            if previous_index is None:
                previous_index = last_gate_index  # sets previous_index to the last gate's index if this is the first time we have a previous gate
            elif previous_index != last_gate_index: # if what we saved as the previous index doesn't match the last gate's o.g. array index stored in the stack
                can_cancel = False # then they can't cancel because the two qubits' stacks' last gate cannot be the same
                can_replace = False
                break

        # should be false by now if any condition failed
        if can_cancel:
            # Remove previous gate from every affected stack (like if its a CNOT, this will remove it form both of its affected qubits)
            for qubit_stack in qubits:
                qubit_stacks[qubit_stack].pop()

            # Mark both gates as canceled
            removed.add(previous_index) # last gate marked
            removed.add(gate_index) # current gate marked

        elif can_replace:
            gates[previous_index]["gate"] = replacement
            removed.add(gate_index) # gets rid of current gate for replacement


        else:

            # Add the current gate to every qubit it acts on--recall we've been holding this gate hostage til we identfiied if it can cancel or not. Can't cancel = add regularly
            for qubit_stack in qubits:
                qubit_stacks[qubit_stack].append(gate_index)

    # Reconstruct the optimized circuit
    optimized_gates = [
        copy.deepcopy(gate_info)
        for i, gate_info in enumerate(gates) if i not in removed
    ]

    print('removed gates:')
    print(removed)
    return {
        "n_qubits": n_qubits,
        "gates": optimized_gates
    }

    return copy.deepcopy(circuit)





def validate_circuit(circuit):
    """Raise ValueError if a circuit violates the public format."""
    if not isinstance(circuit, dict) or set(circuit) != {"n_qubits", "gates"}:
        raise ValueError("Circuit must have exactly 'n_qubits' and 'gates'.")

    n = circuit["n_qubits"]
    if type(n) is not int or not 1 <= n <= 6:
        raise ValueError("n_qubits must be an integer from 1 through 6.")

    gates = copy.deepcopy(circuit["gates"])
    if not isinstance(gates, list):
        raise ValueError("gates must be a list.")

    for i, operation in enumerate(gates):
        if not isinstance(operation, dict) or set(operation) != {"gate", "qubits"}:
            raise ValueError(f"Gate {i}: expected exactly 'gate' and 'qubits'.")

        name = operation["gate"]
        qubits = operation["qubits"]
        if not isinstance(name, str) or name not in ALLOWED_GATES:
            raise ValueError(f"Gate {i}: unsupported gate {name!r}.")
        if not isinstance(qubits, list):
            raise ValueError(f"Gate {i}: qubits must be a list.")

        arity = 2 if name == "CX" else 1
        if len(qubits) != arity:
            raise ValueError(f"Gate {i}: {name} requires {arity} qubit(s).")
        if any(type(q) is not int or not 0 <= q < n for q in qubits):
            raise ValueError(f"Gate {i}: invalid qubit index.")
        if len(set(qubits)) != len(qubits):
            raise ValueError(f"Gate {i}: qubit indices must be distinct.")

    return True


def validate_config(config):
    """Check nonnegative finite resource weights w1, w2, wd."""
    if not isinstance(config, dict) or set(config) != {"w1", "w2", "wd"}:
        raise ValueError("config must have exactly 'w1', 'w2', and 'wd'.")

    for key in ("w1", "w2", "wd"):
        value = config[key]
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError(f"{key} must be a real number.")
        if not math.isfinite(value) or value < 0:
            raise ValueError(f"{key} must be finite and nonnegative.")

    if not any(config[key] > 0 for key in ("w1", "w2", "wd")):
        raise ValueError("At least one weight must be positive.")

    return True


_ROOT2 = math.sqrt(2)
_ONE_QUBIT_MATRICES = {
    "H": np.array([[1, 1], [1, -1]], dtype=complex) / _ROOT2,
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
    "S": np.diag([1, 1j]).astype(complex),
    "SDG": np.diag([1, -1j]).astype(complex),
    "T": np.diag([1, np.exp(1j * np.pi / 4)]).astype(complex),
    "TDG": np.diag([1, np.exp(-1j * np.pi / 4)]).astype(complex),
}


def _full_gate_matrix(n, operation):
    """Matrix for one gate, with qubit 0 as the most significant bit."""
    name = operation["gate"]
    qubits = operation["qubits"]
    if name != "CX":
        result = np.array([[1]], dtype=complex)
        for q in range(n):
            factor = (_ONE_QUBIT_MATRICES[name]
                      if q == qubits[0] else np.eye(2, dtype=complex))
            result = np.kron(result, factor)
        return result

    control, target = qubits
    dimension = 1 << n
    result = np.zeros((dimension, dimension), dtype=complex)
    control_bit = 1 << (n - 1 - control)
    target_bit = 1 << (n - 1 - target)
    for input_state in range(dimension):
        output_state = (input_state ^ target_bit
                        if input_state & control_bit else input_state)
        result[output_state, input_state] = 1
    return result


def circuit_unitary(circuit):
    """Calculate the complete unitary for a supported 1-6 qubit circuit.

    Earlier listed gates act first: [A, B] has unitary U_B @ U_A.
    """
    validate_circuit(circuit)
    n = circuit["n_qubits"]
    result = np.eye(1 << n, dtype=complex)
    for operation in circuit["gates"]:
        result = _full_gate_matrix(n, operation) @ result
    return result


def equivalent_up_to_global_phase(first, second, atol=1e-9):
    """Return True if both circuits implement the same unitary up to phase."""
    validate_circuit(first)
    validate_circuit(second)
    if first["n_qubits"] != second["n_qubits"]:
        return False

    a = circuit_unitary(first)
    b = circuit_unitary(second)
    pivot = np.unravel_index(np.argmax(np.abs(a)), a.shape)
    phase = b[pivot] / a[pivot]
    return (np.isclose(abs(phase), 1.0, atol=atol, rtol=0.0)
            and np.allclose(b, phase * a, atol=atol, rtol=0.0))


def circuit_metrics(circuit):
    """Count gates and minimum depth, scheduling disjoint gates in parallel.

    Every gate takes one time unit. Gates sharing a qubit are serialized.
    """
    validate_circuit(circuit)
    available = [0] * circuit["n_qubits"]
    one_qubit = 0
    two_qubit = 0

    for operation in circuit["gates"]:
        qubits = operation["qubits"]
        end = 1 + max(available[q] for q in qubits)
        for q in qubits:
            available[q] = end
        if operation["gate"] == "CX":
            two_qubit += 1
        else:
            one_qubit += 1

    return {"n_1q": one_qubit, "n_2q": two_qubit, "depth": max(available)}


def circuit_cost(circuit, config):
    """Weighted cost C = w1*N_1q + w2*N_2q + wd*depth."""
    validate_config(config)
    metrics = circuit_metrics(circuit)
    return (config["w1"] * metrics["n_1q"]
            + config["w2"] * metrics["n_2q"]
            + config["wd"] * metrics["depth"])


def evaluate_optimization(original, optimized, config):
    """Return local correctness and resource comparisons for one input."""
    equivalent = equivalent_up_to_global_phase(original, optimized)
    before = circuit_cost(original, config)
    after = circuit_cost(optimized, config)
    return {
        "equivalent": equivalent,
        "input_metrics": circuit_metrics(original),
        "output_metrics": circuit_metrics(optimized),
        "input_cost": before,
        "output_cost": after,
        "cost_reduction": before - after,
    }


if __name__ == "__main__":
    example = {
        "n_qubits": 2,
        "gates": [
            {"gate": "CX", "qubits": [0, 1]},
            {"gate": "Z", "qubits": [0]},
            {"gate": "CX", "qubits": [0, 1]},
        ],
    }
    weights = {"w1": 1.0, "w2": 6.0, "wd": 0.2}
    result = optimize(example, weights)
    print("Your optimizer (initially an unchanged copy):")
    print(result)
    print("Local assessment:")
    print(evaluate_optimization(example, result, weights))

    # A known equivalent circuit to demonstrate the checker:
    expected = {"n_qubits": 2, "gates": [{"gate": "Z", "qubits": [0]}]}
    print("Known simplification (CX-Z-CX -> Z):")
    print(evaluate_optimization(example, expected, weights))
    print()
    print()
    print()


    # example
    example = {
    "n_qubits": 3,
    "gates": [
        {"gate": "H", "qubits": [0]},
        {"gate": "X", "qubits": [1]},
        {"gate": "H", "qubits": [0]},
        {"gate": "CX", "qubits": [1, 2]},
        {"gate": "CX", "qubits": [1, 2]},
        {"gate": "T", "qubits": [0]},
        {"gate": "TDG", "qubits": [0]},
        {"gate": "TDG", "qubits": [0]},
        {"gate": "TDG", "qubits": [0]},
        {"gate": "SDG", "qubits": [0]}
     ]
    }

    weights = {"w1": 1.0, "w2": 6.0, "wd": 0.2}

    result = optimize(example, weights)

    print("Optimized circuit:")
    print(result)

    print("\nEvaluation:")
    print(evaluate_optimization(example, result, weights))